In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR2TOT_G.xpt"
csv_file = DATA_DIR / "DR2TOT_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 DR2TOT_G XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "WTDRD1",
    "WTDR2D",
    "DR2DRSTZ",
    "DR2EXMER",
    "DRABF",
    "DRDINT",
    "DR2DBIH",
    "DR2DAY",
    "DR2LANG",
    "DR2MNRSP",
    "DR2HELPD",
    "DR2TNUMF",
    "DR2TKCAL",
    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",
    "DR2TCHOL",
    "DR2TATOC",
    "DR2TATOA",
    "DR2TRET",
    "DR2TVARA",
    "DR2TACAR",
    "DR2TBCAR",
    "DR2TCRYP",
    "DR2TLYCO",
    "DR2TLZ",
    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",
    "DR2TFOLA",
    "DR2TFA",
    "DR2TFF",
    "DR2TFDFE",
    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",
    "DR2TCALC",
    "DR2TPHOS",
    "DR2TMAGN",
    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSODI",
    "DR2TPOTA",
    "DR2TSELE",
    "DR2TCAFF",
    "DR2TTHEO",
    "DR2TALCO",
    "DR2TMOIS",
    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",
    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",
    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",
    "DR2_300",
    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ",
    "DR2TWS"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "DR2TOT_G column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 9338:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 83:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,338 rows x 83 columns."
)


# ============================================================
# 4. DEFINE GENUINE DIETARY DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# These 50 variables contain genuine fractional values
# in the attached DR2TOT_G.xpt.
#
# DO NOT:
#
#   - round them
#   - convert them to Int64
#   - use float_format when exporting
#
# Genuine values such as:
#
#     0.0
#     1.0
#     1.25
#     76.35
#
# must be allowed to serialize naturally.
#

DECIMAL_COLUMNS = [
    "WTDRD1",
    "WTDR2D",

    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",

    "DR2TATOC",
    "DR2TATOA",

    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",

    "DR2TCHL",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVD",
    "DR2TVK",

    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSELE",

    "DR2TALCO",
    "DR2TMOIS",

    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",

    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",

    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",

    "DR2_320Z",
    "DR2_330Z",
    "DR2BWATZ"
]


if len(DECIMAL_COLUMNS) != 50:

    raise ValueError(
        "Expected exactly 50 genuine decimal variables."
    )


INTEGER_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(INTEGER_COLUMNS) != 33:

    raise ValueError(
        "Expected exactly 33 whole-number variables."
    )


print("\n--- VARIABLE TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    len(DECIMAL_COLUMNS)
)

print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER globally replace it before validating
# the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros before correction:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT values:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact pattern found in attached DR2TOT_G.xpt.
#
# 64 variables
# 48,325 total tiny values
#
# These are validated zero-capable variables based on
# their CDC ranges plus the attached XPT pattern.
#

EXPECTED_TINY_COUNTS = {

    "DR2BWATZ": 4843,

    "DR2DBIH": 1,

    "DR2TACAR": 979,

    "DR2TALCO": 6759,

    "DR2TATOA": 6171,

    "DR2TATOC": 10,

    "DR2TB12A": 4821,

    "DR2TBCAR": 108,

    "DR2TCAFF": 2237,

    "DR2TCARB": 7,

    "DR2TCHL": 8,

    "DR2TCHOL": 88,

    "DR2TCRYP": 809,

    "DR2TFA": 194,

    "DR2TFDFE": 8,

    "DR2TFF": 80,

    "DR2TFIBE": 86,

    "DR2TFOLA": 8,

    "DR2TIRON": 8,

    "DR2TKCAL": 6,

    "DR2TLYCO": 2111,

    "DR2TLZ": 94,

    "DR2TM161": 25,

    "DR2TM181": 10,

    "DR2TM201": 63,

    "DR2TM221": 1582,

    "DR2TMFAT": 8,

    "DR2TNIAC": 9,

    "DR2TP182": 8,

    "DR2TP183": 8,

    "DR2TP184": 3327,

    "DR2TP204": 147,

    "DR2TP205": 1059,

    "DR2TP225": 815,

    "DR2TP226": 985,

    "DR2TPFAT": 8,

    "DR2TPHOS": 6,

    "DR2TPOTA": 6,

    "DR2TPROT": 8,

    "DR2TRET": 63,

    "DR2TS040": 354,

    "DR2TS060": 428,

    "DR2TS080": 129,

    "DR2TS100": 36,

    "DR2TS120": 27,

    "DR2TS140": 10,

    "DR2TS160": 8,

    "DR2TS180": 9,

    "DR2TSELE": 9,

    "DR2TSFAT": 8,

    "DR2TSUGR": 10,

    "DR2TTFAT": 9,

    "DR2TTHEO": 3963,

    "DR2TVARA": 18,

    "DR2TVB1": 9,

    "DR2TVB12": 20,

    "DR2TVB2": 8,

    "DR2TVB6": 9,

    "DR2TVC": 50,

    "DR2TVD": 154,

    "DR2TVK": 15,

    "DR2TZINC": 2,

    "DR2_320Z": 1573,

    "DR2_330Z": 3894
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DR2TOT_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 48325:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total:,}"
    )


if len(EXPECTED_TINY_COUNTS) != 64:

    raise ValueError(
        "Expected tiny artifacts in exactly 64 variables."
    )


print(
    "\nPASS: Exact 48,325-value tiny-zero "
    "pattern across 64 variables validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

#
# Do NOT globally replace every tiny value.
#
# Only restore the exact variables/counts above.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} tiny values, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 48325:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what the corrected
# XPT should contain.
#
# ONLY validated:
#
#     TINY_VALUE -> 0
#
# positions change.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


if remaining_tiny != 0:

    raise ValueError(
        f"Tiny values remain after correction: "
        f"{remaining_tiny:,}"
    )


print(
    "PASS: No tiny XPORT artifacts remain "
    "after restoration."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# This is the CRITICAL Dietary safeguard.
#
# Run AFTER validated zero restoration but BEFORE
# converting anything to Int64.
#
# Expected:
#
#     exactly 50 genuine decimal variables
#

print("\n--- GENUINE DECIMAL CHECK ---")


detected_decimal_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "STOP: Decimal-variable pattern changed.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "\nPASS: Exactly 50 genuine Dietary decimal "
    "variables detected."
)


# ============================================================
# 11. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 12. FORMAT ONLY PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# Only the 33 proven whole-number fields are converted
# to nullable Int64.
#
# The 50 genuine Dietary decimal fields are untouched.
#
# NO .round() is used.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "genuine decimal values."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: 33 whole-number variables "
    "converted to Int64."
)


print(
    "PASS: All 50 genuine Dietary decimal "
    "variables remain unchanged."
)


# ============================================================
# 13. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 14. CDC CHECK - DIETARY DAY 1 SAMPLE WEIGHT
# ============================================================

#
# WTDRD1 remains in the Day-2 file because both Day-1 and
# two-day weights are included.
#
# CDC:
#
# 1852.3028731 - 443805.73786
# n = 8519
# missing = 819
#

s = df["WTDRD1"]


if not (
    int(
        s.notna()
        .sum()
    )
    == 8519
    and
    int(
        s.isna()
        .sum()
    )
    == 819
    and
    np.isclose(
        s.min(),
        1852.3028731,
        rtol=0,
        atol=1e-6
    )
    and
    np.isclose(
        s.max(),
        443805.73786,
        rtol=0,
        atol=1e-5
    )
):

    raise ValueError(
        "WTDRD1 differs from CDC."
    )


print(
    "PASS: WTDRD1 matches CDC."
)


# ============================================================
# 15. CDC CHECK - TWO-DAY SAMPLE WEIGHT
# ============================================================

#
# WTDR2D:
#
# 1531.3308619 - 499271.52935
# n = 7605
# missing = 1733
#

s = df["WTDR2D"]


if not (
    int(
        s.notna()
        .sum()
    )
    == 7605
    and
    int(
        s.isna()
        .sum()
    )
    == 1733
    and
    np.isclose(
        s.min(),
        1531.3308619,
        rtol=0,
        atol=1e-6
    )
    and
    np.isclose(
        s.max(),
        499271.52935,
        rtol=0,
        atol=1e-5
    )
):

    raise ValueError(
        "WTDR2D differs from CDC."
    )


print(
    "PASS: WTDR2D matches CDC."
)


# ============================================================
# 16. CDC CHECK - DR2DRSTZ
# ============================================================

#
# CDC:
#
# 1 = Reliable                7,496
# 2 = Not reliable               56
# 4 = Breast milk               109
# 5 = Not done                1,677
#

check_frequency(
    "DR2DRSTZ",
    {
        1: 7496,
        2: 56,
        4: 109,
        5: 1677
    },
    0
)


# ============================================================
# 17. CDC CHECK - DRDINT
# ============================================================

#
# 1 = Day 1 only       914
# 2 = Day 1 + Day 2   7605
# missing              819
#

check_frequency(
    "DRDINT",
    {
        1: 914,
        2: 7605
    },
    819
)


# ============================================================
# 18. CDC CHECK - DR2DBIH
# ============================================================

#
# Number of days between Day-2 intake and family interview.
#
# CDC:
#
# -38 to 178
# n = 7638
# missing = 1700
#
# CDC states 0 means the family interview occurred
# on the same date as the dietary intake day.
#
# Attached XPT contains exactly 1 tiny-zero artifact.
#

s = df["DR2DBIH"]


print("\n--- DR2DBIH ---")


print(
    "Minimum:",
    s.min()
)


print(
    "Maximum:",
    s.max()
)


print(
    "Zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    s.min() == -38
    and
    s.max() == 178
    and
    int(
        s.notna()
        .sum()
    )
    == 7638
    and
    int(
        s.isna()
        .sum()
    )
    == 1700
    and
    int(
        s.eq(0)
        .sum()
    )
    == 1
):

    raise ValueError(
        "DR2DBIH differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: DR2DBIH matches CDC."
)


# ============================================================
# 19. CDC CHECK - DAY OF WEEK
# ============================================================

check_frequency(
    "DR2DAY",
    {
        1: 1257,
        2: 1556,
        3: 1455,
        4: 1771,
        5: 726,
        6: 716,
        7: 180
    },
    1677
)


# ============================================================
# 20. CDC CHECK - NUMBER OF FOODS
# ============================================================

#
# CDC:
#
# 1-46 = 7,605
# missing = 1,733
#

s = df["DR2TNUMF"]


if not (
    s.min() == 1
    and
    s.max() == 46
    and
    int(
        s.notna()
        .sum()
    )
    == 7605
    and
    int(
        s.isna()
        .sum()
    )
    == 1733
):

    raise ValueError(
        "DR2TNUMF differs from CDC."
    )


print(
    "PASS: DR2TNUMF matches CDC."
)


# ============================================================
# 21. CDC CHECK - ENERGY
# ============================================================

#
# DR2TKCAL is a released whole-number kcal field.
#
# CDC:
#
# 0-8213
# n = 7496
# missing = 1842
#
# Attached XPT:
#
# 6 tiny artifacts -> 6 legitimate zeros
#

s = df["DR2TKCAL"]


if not (
    s.min() == 0
    and
    s.max() == 8213
    and
    int(
        s.notna()
        .sum()
    )
    == 7496
    and
    int(
        s.isna()
        .sum()
    )
    == 1842
    and
    int(
        s.eq(0)
        .sum()
    )
    == 6
):

    raise ValueError(
        "DR2TKCAL differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: DR2TKCAL matches CDC."
)


# ============================================================
# 22. CDC CHECK - PROTEIN
# ============================================================

#
# Genuine Dietary decimal.
#
# CDC:
#
# 0-355.95 gm
# n = 7496
# missing = 1842
#
# Attached XPT:
#
# 8 tiny values -> 8 numeric 0.0 values.
#

s = df["DR2TPROT"]


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        355.95,
        rtol=0,
        atol=1e-10
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 7496
    and
    int(
        s.isna()
        .sum()
    )
    == 1842
    and
    int(
        s.eq(0)
        .sum()
    )
    == 8
):

    raise ValueError(
        "DR2TPROT differs from CDC."
    )


print(
    "PASS: DR2TPROT matches CDC."
)


# ============================================================
# 23. CDC CHECK - ALCOHOL
# ============================================================

#
# Genuine decimal.
#
# CDC:
#
# 0-325.2 gm
# n = 7496
# missing = 1842
#
# Attached XPT:
#
# 6,759 validated zeros.
#

s = df["DR2TALCO"]


print("\n--- DR2TALCO ---")


print(
    "Zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        325.2,
        rtol=0,
        atol=1e-10
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 7496
    and
    int(
        s.isna()
        .sum()
    )
    == 1842
    and
    int(
        s.eq(0)
        .sum()
    )
    == 6759
):

    raise ValueError(
        "DR2TALCO differs from CDC."
    )


print(
    "PASS: DR2TALCO matches CDC."
)


# ============================================================
# 24. CDC CHECK - AMOUNT VS USUAL
# ============================================================

check_frequency(
    "DR2_300",
    {
        1: 427,
        2: 6263,
        3: 935,
        9: 36
    },
    1677
)


# ============================================================
# 25. CDC CHECK - TOTAL PLAIN WATER
# ============================================================

#
# DR2_320Z is a genuine decimal variable.
#
# CDC:
#
# 0-10665 gm
# n = 7605
# missing = 1733
#
# Attached XPT:
#
# 1,573 zero artifacts.
#

s = df["DR2_320Z"]


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        10665,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 7605
    and
    int(
        s.isna()
        .sum()
    )
    == 1733
    and
    int(
        s.eq(0)
        .sum()
    )
    == 1573
):

    raise ValueError(
        "DR2_320Z differs from CDC."
    )


print(
    "PASS: DR2_320Z matches CDC."
)


# ============================================================
# 26. CDC CHECK - TOTAL TAP WATER
# ============================================================

#
# Genuine decimal.
#
# CDC:
#
# 0-10665 gm
# n = 7605
# missing = 1733
#
# Attached XPT:
#
# 3,894 zeros.
#

s = df["DR2_330Z"]


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        10665,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 7605
    and
    int(
        s.isna()
        .sum()
    )
    == 1733
    and
    int(
        s.eq(0)
        .sum()
    )
    == 3894
):

    raise ValueError(
        "DR2_330Z differs from CDC."
    )


print(
    "PASS: DR2_330Z matches CDC."
)


# ============================================================
# 27. CDC CHECK - TOTAL BOTTLED WATER
# ============================================================

#
# Genuine decimal.
#
# CDC:
#
# 0-10000 gm
# n = 7605
# missing = 1733
#
# Attached XPT:
#
# 4,843 zeros.
#

s = df["DR2BWATZ"]


if not (
    np.isclose(
        s.min(),
        0,
        rtol=0,
        atol=0
    )
    and
    np.isclose(
        s.max(),
        10000,
        rtol=0,
        atol=0
    )
    and
    int(
        s.notna()
        .sum()
    )
    == 7605
    and
    int(
        s.isna()
        .sum()
    )
    == 1733
    and
    int(
        s.eq(0)
        .sum()
    )
    == 4843
):

    raise ValueError(
        "DR2BWATZ differs from CDC."
    )


print(
    "PASS: DR2BWATZ matches CDC."
)


# ============================================================
# 28. CDC CHECK - TAP WATER SOURCE
# ============================================================

#
# CDC:
#
# 1  Community supply       4935
# 2  Well/rain cistern       595
# 3  Spring                   79
# 4  Don't drink tap water  1639
# 99 Don't know              413
# Missing                   1677
#

check_frequency(
    "DR2TWS",
    {
        1: 4935,
        2: 595,
        3: 79,
        4: 1639,
        99: 413
    },
    1677
)


# ============================================================
# 29. VALIDATED ZERO COUNT CHECK
# ============================================================

#
# Check every corrected variable has the expected
# number of numeric zeros.
#
# Because the raw attached XPT contained zero ordinary
# numeric zeros, the expected post-correction zero count
# equals the tiny-artifact count for these variables.
#

print("\n--- VALIDATED ZERO COUNTS ---")


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    actual_zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if actual_zero_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,} zeros, "
            f"found {actual_zero_count:,}."
        )


print(
    "PASS: All 64 corrected variables have "
    "the expected zero counts."
)


# ============================================================
# 30. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# CRITICAL:
#
# The ONLY numeric differences from pandas' original
# XPT should be:
#
#     48,325 validated TINY_VALUE -> 0 restorations
#
# Nothing else may change.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 83 variables match the expected "
    "corrected XPT exactly."
)


# ============================================================
# 31. EXPORT CSV
# ============================================================

#
# CRITICAL FOR DIETARY:
#
# NO:
#
#     float_format=
#
# NO:
#
#     rounding
#
# The 50 genuine Dietary decimal variables are allowed
# to serialize naturally.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 32. RE-READ WITH ROUND-TRIP FLOAT PARSER
# ============================================================

#
# float_precision="round_trip"
#
# allows exported decimal text to be parsed back to
# the same binary floating-point value.
#

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed after export."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 33. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV numeric differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 83 CSV variables round-trip "
    "to exact corrected XPT numeric values."
)


# ============================================================
# 34. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 35. READ RAW CSV AS TEXT FOR FORMAT CHECKING
# ============================================================

#
# Numeric fidelity was checked above.
#
# Now read as strings so we can inspect formatting
# without pandas coercing fields back to floats.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 36. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in CSV."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 37. .0 CHECK - WHOLE-NUMBER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT:
#
#     csv_text.count(".0")
#
# across the whole Dietary CSV.
#
# Genuine Dietary decimals are allowed to contain:
#
#     0.0
#     1.0
#     6.0
#
# Only the 33 proven INTEGER_COLUMNS are checked.
#

print("\n--- WHOLE-NUMBER .0 FORMAT CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in "
        "whole-number fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in "
    "whole-number variables."
)


# ============================================================
# 38. DECIMAL .0 VALUES ARE LEGITIMATE
# ============================================================

#
# Informational check only.
#
# Genuine decimal columns may legitimately serialize
# with ".0".
#
# We intentionally leave them AS IS.
#

decimal_dot_zero_counts = {}


for col in DECIMAL_COLUMNS:

    count = int(
        (
            raw_csv_df[col].ne("")
            &
            raw_csv_df[col].str.endswith(".0")
        )
        .sum()
    )


    if count > 0:

        decimal_dot_zero_counts[col] = count


print(
    "\nDecimal fields containing legitimate '.0' values:",
    len(decimal_dot_zero_counts)
)


print(
    "PASS: Genuine Dietary '.0' values were not removed."
)


# ============================================================
# 39. EXACT DIETARY DECIMAL FIDELITY
# ============================================================

#
# This directly verifies every genuine decimal column
# against the corrected XPT values.
#

print("\n--- DIETARY DECIMAL FIDELITY ---")


decimal_failures = []


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        decimal_failures.append(
            col
        )


if decimal_failures:

    raise ValueError(
        "Dietary decimal fidelity failed:\n"
        + str(decimal_failures)
    )


print(
    "PASS: All 50 genuine Dietary decimal "
    "variables preserved exactly."
)


# ============================================================
# 40. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 41. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid and contains no .0."
)


# ============================================================
# 42. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "DR2TOT_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Variables containing tiny artifacts:",
    len(EXPECTED_TINY_COUNTS)
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine Dietary decimal variables:",
    len(DECIMAL_COLUMNS)
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0 in whole-number fields: "
    "0 expected"
)


print(
    "Dietary decimal values altered: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 DR2TOT_G XPT -> CSV
Rows: 9,338
Columns: 83
PASS: Structure = 9,338 rows x 83 columns.

--- VARIABLE TYPE PLAN ---
Genuine decimal variables: 50
Whole-number variables: 33

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros before correction: 0
Tiny XPORT values: 48,325

Tiny values by variable:
DR2TALCO    6759
DR2TATOA    6171
DR2BWATZ    4843
DR2TB12A    4821
DR2TTHEO    3963
DR2_330Z    3894
DR2TP184    3327
DR2TCAFF    2237
DR2TLYCO    2111
DR2TM221    1582
DR2_320Z    1573
DR2TP205    1059
DR2TP226     985
DR2TACAR     979
DR2TP225     815
DR2TCRYP     809
DR2TS060     428
DR2TS040     354
DR2TFA       194
DR2TVD       154
DR2TP204     147
DR2TS080     129
DR2TBCAR     108
DR2TLZ        94
DR2TCHOL      88
DR2TFIBE      86
DR2TFF        80
DR2TRET       63
DR2TM201      63
DR2TVC        50
DR2TS100      36
DR2TS120      27
DR2TM161      25
DR2TVB12      20
DR2TVARA      18
DR2TVK        15
DR2TATOC      10
DR2TS140      10
DR2TM181      10
DR2TSUGR      10
DR